## Imports and config

In [ ]:
# imports & basic config.

# Phase 01 — Chunk 01: Imports & Basic Config
import os
import numpy as np
import pandas as pd
import datetime as dt

import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Configure Plotly for notebook
pio.renderers.default = "notebook"  # or "iframe_connected" if notebook gives issues

# Pandas display options (clean output)
pd.set_option("display.float_format", "{:.6f}".format)

print("Chunk 01 executed successfully.")

In [ ]:
# Chunk 2: choose underlying and date range
UNDERLYING = "SPY"
AS_OF = pd.Timestamp.today().normalize()   # or set a date like pd.Timestamp("2025-11-01")
print("Underlying:", UNDERLYING, "| As of:", AS_OF.date())

In [ ]:
# Chunk 3: fetch available expiries
tk = yf.Ticker(UNDERLYING)
expiries = tk.options  # list of expiry strings like '2025-11-21'
print("Number of expiries available:", len(expiries))
print("First 8 expiries:", expiries[:8])

In [ ]:
# Chunk 4: function to fetch an option chain (calls + puts) for a given expiry
def fetch_chain_for_expiry(ticker_obj, expiry):
    # returns combined DataFrame with 'contractSymbol', 'type', 'strike', 'lastPrice', 'bid', 'ask', 'volume', 'openInterest', 'impliedVol'
    chain = ticker_obj.option_chain(expiry)
    calls = chain.calls.copy()
    calls['type'] = 'call'
    puts = chain.puts.copy()
    puts['type'] = 'put'
    df = pd.concat([calls, puts], ignore_index=True, sort=False)
    df['expiry'] = pd.to_datetime(expiry)
    return df

# example for the nearest expiry
sample_expiry = expiries[0]
sample_df = fetch_chain_for_expiry(tk, sample_expiry)
sample_df.head()

In [ ]:
# Chunk 5: fetch many expiries (be careful with rate limits). Limit to next N expiries for speed.

N_EXPIRIES = 8   # start with 6-12 expiries; increase later
exp_slice = expiries[:N_EXPIRIES]

all_dfs = []
for e in exp_slice:
    try:
        dfe = fetch_chain_for_expiry(tk, e)
        all_dfs.append(dfe)
    except Exception as exc:
        print("Failed expiry", e, exc)

opts = pd.concat(all_dfs, ignore_index=True, sort=False)
print("Total option rows:", len(opts))
opts.info()

In [ ]:
# Chunk 6: cleaning + compute mid price + maturity in years
# Compute mid price from bid/ask; fallback to lastPrice if bid/ask absent


opts['bid'] = pd.to_numeric(opts['bid'], errors='coerce')
opts['ask'] = pd.to_numeric(opts['ask'], errors='coerce')
opts['lastPrice'] = pd.to_numeric(opts['lastPrice'], errors='coerce')
opts['mid'] = opts[['bid', 'ask']].mean(axis=1)
opts['mid'] = opts['mid'].fillna(opts['lastPrice'])  # fallback

# time to expiry in year fraction
today = AS_OF
opts['T_days'] = (opts['expiry'] - today).dt.days
opts = opts[opts['T_days'] > 0].copy()
opts['T'] = opts['T_days'] / 365.0

# filter illiquid strikes (optional thresholds)
opts['volume'] = pd.to_numeric(opts.get('volume', 0), errors='coerce').fillna(0)
opts['openInterest'] = pd.to_numeric(opts.get('openInterest', 0), errors='coerce').fillna(0)
liquid_mask = (opts['openInterest'] > 0) | (opts['volume'] > 0)
opts = opts[liquid_mask].reset_index(drop=True)

print("After liquidity filter rows:", len(opts))
opts[['type', 'strike', 'expiry', 'T', 'mid', 'volume', 'openInterest']].head()


# Interpretation: 'mid price' is best to compare to model price. We filter out options with zero OI and zero volume to avoid stale quotes.

# Checklist: mid has meaningful values; T > 0; verify filters didn’t wipe out all data.


In [ ]:
# Chunk 7: underlying price & risk-free rate (approx)


spot = tk.history(period="1d")['Close'].iloc[-1]
print("SPY spot:", spot)

# Simple risk-free approximation: use short-term Treasury rate or set to small number, e.g., 0.025
# For more realism, you can pull daily Fed funds from FRED; we keep a placeholder for now.

r = 0.03  # 3% annualized; replace with fetched rates if you want exact
opts['spot'] = spot
opts['r'] = r

# Interpretation: spot price is needed for BS pricing. Use a proxy r for now (we can fetch actual treasury yields from FRED later).
# Checklist: spot printed and filled across opts.

In [ ]:
# Chunk 8: sanitize mid prices: remove negative / zero or too-large values


opts = opts[(opts['mid'] > 0.0001) & (opts['strike'] > 0)]

# also remove extreme T near zero if needed (very short-dated options can be noisy)


opts = opts[opts['T'] >= 1/365.0].copy()  # at least 1 day
print("Rows after price sanity:", len(opts))

In [ ]:
# === Chunk 9 (complete, safe, Plotly-based) ===
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# --- Safe save helper (tries preferred_dir, then ./plots, then cwd) ---
def safe_write_html(fig, filename, preferred_dir="/mnt/data"):
    # Try preferred dir first (useful in sandboxes); then ./plots; then cwd
    candidates = [preferred_dir, "plots", os.getcwd()]
    last_exc = None
    for d in candidates:
        try:
            if d and not os.path.exists(d):
                os.makedirs(d, exist_ok=True)
            out_path = os.path.join(d if d else "", filename)
            fig.write_html(out_path)
            print(f"Saved: {out_path}")
            return out_path
        except Exception as e:
            last_exc = e
            # try next candidate
            continue
    # if we reach here, raise the last exception
    raise RuntimeError(f"Failed to save {filename}. Last error: {last_exc}")

# --- Ensure 'opts' exists ---
try:
    opts  # reference to the options DataFrame you built earlier
except NameError:
    raise RuntimeError("DataFrame `opts` not found. Run previous Phase 1 chunks to build `opts` before this cell.")

# --- 1) Show columns & head for quick debug ---
print("Columns present in opts:", list(opts.columns))
display(opts.head())

# --- 2) Compute / ensure numeric strike and spot columns exist ---
# Ensure strike numeric
if 'strike' not in opts.columns:
    raise RuntimeError("Column 'strike' missing from opts. Check earlier fetch steps.")
opts['strike'] = pd.to_numeric(opts['strike'], errors='coerce')

# Ensure spot: prefer opts['spot'], else fetch from ticker tk if available
if 'spot' not in opts.columns or opts['spot'].isnull().all():
    try:
        spot = tk.history(period="1d")['Close'].iloc[-1]
        opts['spot'] = float(spot)
        print(f"Filled opts['spot'] with fetched spot: {spot:.4f}")
    except Exception:
        raise RuntimeError("Missing underlying spot price. Ensure `opts['spot']` exists or `tk` ticker object is available.")

# --- 3) Compute moneyness if missing or recompute to be safe ---
opts['moneyness'] = pd.to_numeric(opts.get('moneyness'), errors='coerce')
missing_before = opts['moneyness'].isna().sum()
if missing_before > 0:
    opts['moneyness'] = opts['strike'] / opts['spot']
    print(f"Computed moneyness for {len(opts)} rows (filled {missing_before} missing entries).")
else:
    # still force numeric conversion
    opts['moneyness'] = pd.to_numeric(opts['moneyness'], errors='coerce')
    print("moneyness existed — ensured numeric dtype.")

# --- 4) Clip mid price for plotting visibility & ensure numeric ---
opts['mid'] = pd.to_numeric(opts.get('mid'), errors='coerce')
opts['mid_clipped'] = opts['mid'].clip(upper=50)  # prevents extreme tails from dominating visuals

# --- 5) Diagnostics: counts, T range, moneyness stats ---
print("\nDiagnostic summary:")
print("Total rows:", len(opts))
print("Rows with valid mid price:", opts['mid'].notna().sum())
print("Rows with valid moneyness:", opts['moneyness'].notna().sum())
print("T (days) min/max (sample):", opts.get('T_days').min() if 'T_days' in opts.columns else 'N/A',
      "/", opts.get('T_days').max() if 'T_days' in opts.columns else 'N/A')
display(opts['moneyness'].describe())

# --- 6) Prepare data for plotting: drop NaNs for the relevant columns ---
plot_df = opts.dropna(subset=['mid_clipped', 'moneyness']).copy()
if plot_df.empty:
    raise RuntimeError("No valid rows to plot after dropping NaNs. Investigate opts['mid'] and opts['moneyness'].")

# --- 7) Plotly histogram: Mid price distribution ---
fig1 = px.histogram(plot_df, x='mid_clipped', nbins=60,
                    title="Mid Price Distribution (clipped at 50)",
                    labels={'mid_clipped': 'Mid Price (clipped at 50)'},
                    marginal="box")
fig1.update_layout(bargap=0.02)
fig1.update_traces(marker=dict(opacity=0.8))
fig1.show()

# Save safely
safe_write_html(fig1, "spy_mid_price_distribution.html")

# --- 8) Plotly histogram: Moneyness distribution ---
fig2 = px.histogram(plot_df, x='moneyness', nbins=60,
                    title="Moneyness Distribution (strike / spot)",
                    labels={'moneyness': 'Moneyness (strike/spot)'},
                    marginal="box")
fig2.update_layout(bargap=0.02)
fig2.update_traces(marker=dict(opacity=0.8))
fig2.show()

# Save safely
safe_write_html(fig2, "spy_moneyness_distribution.html")

# --- 9) Expiry summary table (interactive) ---
summary = plot_df.groupby('expiry').agg(
    strike_min=('strike','min'),
    strike_max=('strike','max'),
    moneyness_min=('moneyness','min'),
    moneyness_max=('moneyness','max'),
    count=('strike','count')
).reset_index().sort_values('expiry')

table_fig = go.Figure(data=[go.Table(
    header=dict(values=list(summary.columns),
                fill_color='grey', font=dict(color='white', size=11)),
    cells=dict(values=[summary[c].astype(str) for c in summary.columns],
               align='left')
)])
table_fig.update_layout(title="Expiry → Strike / Moneyness Summary")
table_fig.show()
safe_write_html(table_fig, "spy_expiry_summary_table.html")

print("\nChunk 9 complete — plotted histograms and expiry summary. Files saved (if permitted) to preferred folders.")

## Black–Scholes baseline & Implied Volatility.

In [ ]:
# A: Imports + BS pricer.


import math
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq
import plotly.express as px

# Black-Scholes price (scalar)
def bs_price(spot, K, T, r, sigma, option_type='call'):
    if T <= 0 or sigma <= 0:
        return float(max(spot - K, 0.0) if option_type == 'call' else max(K - spot, 0.0))
    d1 = (math.log(spot / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    if option_type == 'call':
        price = spot * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * math.exp(-r * T) * norm.cdf(-d2) - spot * norm.cdf(-d1)
    return float(price)

# Safe HTML writer (re-uses the helper pattern from Phase 1)
import os
def safe_write_html(fig, filename, preferred_dir="/mnt/data"):
    candidates = [preferred_dir, "plots", os.getcwd()]
    last_exc = None
    for d in candidates:
        try:
            if d and not os.path.exists(d):
                os.makedirs(d, exist_ok=True)
            out_path = os.path.join(d if d else "", filename)
            fig.write_html(out_path)
            print(f"Saved: {out_path}")
            return out_path
        except Exception as e:
            last_exc = e
            continue
    raise RuntimeError(f"Failed to save {filename}. Last error: {last_exc}")

print("Chunk A: Helpers ready.")

# Interpretation / Checklist

#  -> Provides bs_price() to compute BS price for one option.
#  -> safe_write_html() saves plots robustly.

In [ ]:
# B: Robust implied-vol solver for a single option


import numpy as np

def bs_implied_vol_single(market_price, spot, K, T, r, option_type='call', tol=1e-6, maxiter=100):
    # intrinsic check
    intrinsic = max(0.0, (spot - K) if option_type == 'call' else (K - spot))
    if market_price <= intrinsic + 1e-12:
        return 1e-12
    # root function
    def f(sigma):
        return bs_price(spot, K, T, r, sigma, option_type) - market_price
    # bracket
    low, high = 1e-6, 5.0
    try:
        fl, fh = f(low), f(high)
        if fl * fh > 0:
            # expand upper bound once
            high = 10.0
            fh = f(high)
            if fl * fh > 0:
                return np.nan
        iv = brentq(f, low, high, xtol=tol, maxiter=maxiter)
        return float(iv)
    except Exception:
        return np.nan

# quick local sanity test (optional)
# print(bs_implied_vol_single(2.0, 400.0, 395.0, 30/365, 0.03, 'call'))

print("B: IV solver ready.")

# Interpretation / Checklist

# 	-> Solves IV for a single option; returns np.nan if solver fails to bracket.

In [ ]:
# C: Apply Implied Volatility (IV) solver to one expiry.


# Chunk C: compute IVs for a single expiry subset (iterative; safe for debugging)

# Choose an expiry index (0 = nearest)
expiries = sorted(opts['expiry'].unique())
if not expiries:
    raise RuntimeError("No expiries found. Ensure Phase 1 completed and opts exists.")

expiry_idx = 0
demo_expiry = expiries[expiry_idx]
print("Selected demo expiry:", demo_expiry)

# Subset and basic numeric coercion
demo_mask = (opts['expiry'] == demo_expiry)
demo_df = opts[demo_mask].copy().reset_index(drop=True)

# Optional: limit rows to speed up first run
MAX_ROWS = 300
if len(demo_df) > MAX_ROWS:
    demo_df = demo_df.iloc[:MAX_ROWS].copy()

# Ensure numeric columns
for col in ['strike','mid','T','spot','r']:
    if col in demo_df.columns:
        demo_df[col] = pd.to_numeric(demo_df[col], errors='coerce')
    else:
        raise RuntimeError(f"Column '{col}' missing from data. Re-run Phase 1.")

print(f"Computing IV for {len(demo_df)} rows (expiry {demo_expiry}) ...")
ivs = []
bs_prices = []
for i, row in demo_df.iterrows():
    S = float(row['spot'])
    K = float(row['strike'])
    T = float(row['T'])
    r = float(row['r']) if not np.isnan(row['r']) else 0.03
    mp = float(row['mid'])
    typ = row['type'] if 'type' in row else 'call'
    iv = bs_implied_vol_single(mp, S, K, T, r, option_type=typ)
    price_bs = bs_price(S, K, T, r, iv if not np.isnan(iv) else 0.2, option_type=typ)
    ivs.append(iv)
    bs_prices.append(price_bs)
    if (i+1) % 50 == 0:
        print(f"  processed {i+1}/{len(demo_df)}")

demo_df['iv_calc'] = ivs
demo_df['bs_price'] = bs_prices
print("Chunk C: IV computation finished. NaN IV count:", demo_df['iv_calc'].isna().sum())

In [ ]:
# D: Merge results back to main opts, diagnostics and table.


# Chunk D: merge demo results back into opts and show diagnostics
# Merge demo results by contractSymbol (safe)
if 'contractSymbol' not in demo_df.columns:
    raise RuntimeError("contractSymbol missing in demo_df; cannot merge safely.")

# Keep a small backup
opts_before = opts.shape[0]
opts = opts.merge(demo_df[['contractSymbol','iv_calc','bs_price']], on='contractSymbol', how='left')

print(f"Chunk D: merged iv_calc & bs_price into opts (rows before {opts_before}, after {opts.shape[0]})")
# Quick head of merged rows for the demo expiry
display(opts[opts['expiry']==demo_expiry][['contractSymbol','strike','mid','iv_calc','bs_price']].head(10))

# Diagnostic stats
print("IV stats (demo):")
print(opts[opts['expiry']==demo_expiry]['iv_calc'].describe())
print("Mean abs(bs price - mid) (demo):",
      np.nanmean(np.abs(opts[opts['expiry']==demo_expiry]['bs_price'] - opts[opts['expiry']==demo_expiry]['mid'])))

In [ ]:
# E: Plot IV smiles and save.


# Chunk E: plot IV smile for demo expiry using Plotly
plot_df = opts[(opts['expiry']==demo_expiry) & opts['iv_calc'].notna()].copy()
if plot_df.empty:
    print("No IVs available to plot for this expiry. Check earlier chunks.")
else:
    fig = px.scatter(plot_df, x='strike', y='iv_calc', color='mid', size='mid',
                     title=f"IV Smile — Expiry {demo_expiry.date()} (count {len(plot_df)})",
                     labels={'iv_calc':'Implied Vol', 'strike':'Strike', 'mid':'Mid Price'})
    fig.update_traces(marker=dict(opacity=0.75))
    fig.show()
    # safe save
    safe_write_html(fig, f"spy_iv_smile_{demo_expiry.date()}.html")
    print("Chunk E: IV smile plotted and saved (if permitted).")

In [ ]:
spot = demo_df['spot'].iloc[0]

filtered = demo_df[
    (demo_df['strike'] > 0.85 * spot) &
    (demo_df['strike'] < 1.15 * spot) &
    (demo_df['iv_calc'] < 2) &
    (demo_df['iv_calc'] > 0.01)
].copy()

fig = px.scatter(filtered,
                 x='strike',
                 y='iv_calc',
                 color='mid',
                 size='mid',
                 title="Filtered IV Smile (Near-ATM Region)",
                 labels={'iv_calc': 'Implied Vol', 'strike': 'Strike'})
fig.show()

## Binomial Tree Pricing (CRR model)

In [ ]:
# === Chunk F: Binomial Tree Pricing (CRR model) ===

""""
✅ What this does

•	Builds a binomial tree with steps time slices
•	Computes payoff at maturity
•	Discounts backwards to today

"""
import numpy as np

def binomial_price(spot, K, T, r, sigma, steps=100, option_type='call'):
    """
    Cox-Ross-Rubinstein Binomial Tree Pricing (European option)
    """

    # Time step
    dt = T / steps

    # Up and Down factors
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u

    # Risk-neutral probability
    p = (np.exp(r * dt) - d) / (u - d)

    # Ensure probability is valid
    if p < 0 or p > 1:
        return np.nan

    # Initialize asset prices at maturity
    prices = np.array([spot * (u ** j) * (d ** (steps - j)) for j in range(steps + 1)])

    # Payoff at maturity
    if option_type == 'call':
        values = np.maximum(prices - K, 0)
    else:
        values = np.maximum(K - prices, 0)

    # Backward induction
    discount = np.exp(-r * dt)

    for _ in range(steps):
        values = discount * (p * values[1:] + (1 - p) * values[:-1])

    return values[0]

print("Binomial pricing function ready.")

In [ ]:
# === Chunk G: Quick test vs Black-Scholes ===

# Take one sample option from demo_df
sample = demo_df.iloc[10]

S = sample['spot']
K = sample['strike']
T = sample['T']
r = sample['r']
sigma = sample['iv_calc'] if not np.isnan(sample['iv_calc']) else 0.2

bs_val = bs_price(S, K, T, r, sigma, option_type='call')
bin_val = binomial_price(S, K, T, r, sigma, steps=200, option_type='call')

print("Black-Scholes Price :", round(bs_val, 4))
print("Binomial Price      :", round(bin_val, 4))
print("Difference          :", round(abs(bs_val - bin_val), 6))

In [ ]:
# === Chunk H: Apply Binomial Pricing to dataset ===

bin_prices = []

for i, row in demo_df.iterrows():
    S = row['spot']
    K = row['strike']
    T = row['T']
    r = row['r']
    sigma = row['iv_calc']

    if np.isnan(sigma):
        bin_prices.append(np.nan)
        continue

    price = binomial_price(S, K, T, r, sigma, steps=100, option_type=row['type'])
    bin_prices.append(price)

demo_df['binomial_price'] = bin_prices

print("Binomial pricing added.")

In [ ]:
# === Chunk I: Model comparison ===

demo_df['bs_error'] = demo_df['bs_price'] - demo_df['mid']
demo_df['bin_error'] = demo_df['binomial_price'] - demo_df['mid']

print("Average absolute error (BS):", np.nanmean(np.abs(demo_df['bs_error'])))
print("Average absolute error (Binomial):", np.nanmean(np.abs(demo_df['bin_error'])))

In [ ]:
# === Chunk J: Error visualization ===

import plotly.express as px

plot_df = demo_df.dropna(subset=['bin_error'])

fig = px.scatter(
    plot_df,
    x='strike',
    y='bin_error',
    color='mid',
    size='mid',
    title="Binomial Pricing Error vs Market",
    labels={'bin_error': 'Pricing Error', 'strike': 'Strike'}
)

fig.show()
safe_write_html(fig, "binomial_error_plot.html")

## Monte Carlo Pricing Engine [ K → L → M → N → O (and optional P) ]

In [ ]:
# === Chunk K: Monte Carlo pricing engine (European) ===

import numpy as np

def mc_price(spot, K, T, r, sigma, n_paths=100_000, option_type='call', seed=42, antithetic=True):
    """
    Monte Carlo pricer for European options under GBM with optional antithetic variates.
    """
    if T <= 0 or sigma <= 0 or spot <= 0 or K <= 0:
        return np.nan

    rng = np.random.default_rng(seed)

    # draw half if using antithetic
    if antithetic:
        half = n_paths // 2
        z = rng.standard_normal(half)
        z = np.concatenate([z, -z])
        if len(z) < n_paths:  # if odd, add one more
            z = np.concatenate([z, rng.standard_normal(1)])
    else:
        z = rng.standard_normal(n_paths)

    drift = (r - 0.5 * sigma**2) * T
    diffusion = sigma * np.sqrt(T) * z
    ST = spot * np.exp(drift + diffusion)

    if option_type == 'call':
        payoffs = np.maximum(ST - K, 0.0)
    else:
        payoffs = np.maximum(K - ST, 0.0)

    price = np.exp(-r * T) * np.mean(payoffs)

    # standard error (for diagnostics)
    std_err = np.exp(-r * T) * np.std(payoffs) / np.sqrt(len(payoffs))
    return price, std_err

print("Monte Carlo pricer ready.")

In [ ]:
# === Chunk L: Test MC vs Black-Scholes on one option ===

sample = demo_df.iloc[10]

S = sample['spot']
K = sample['strike']
T = sample['T']
r = sample['r']
sigma = sample['iv_calc'] if not np.isnan(sample['iv_calc']) else 0.2
opt_type = sample['type']

bs_val = bs_price(S, K, T, r, sigma, option_type=opt_type)
mc_val, mc_se = mc_price(S, K, T, r, sigma, n_paths=200_000, option_type=opt_type, seed=7)

print("Black-Scholes Price :", round(bs_val, 4))
print("Monte Carlo Price   :", round(mc_val, 4))
print("Std Error (MC)      :", round(mc_se, 6))
print("Abs Diff            :", round(abs(bs_val - mc_val), 4))

In [ ]:
# === Chunk M: Apply Monte Carlo pricing to dataset ===

mc_prices = []
mc_ses = []

for _, row in demo_df.iterrows():
    S = row['spot']
    K = row['strike']
    T = row['T']
    r = row['r']
    sigma = row['iv_calc']
    opt_type = row['type']

    if np.isnan(sigma) or T <= 0:
        mc_prices.append(np.nan)
        mc_ses.append(np.nan)
        continue

    price, se = mc_price(S, K, T, r, sigma, n_paths=100_000, option_type=opt_type, seed=123)
    mc_prices.append(price)
    mc_ses.append(se)

demo_df['mc_price'] = mc_prices
demo_df['mc_se'] = mc_ses

print("Monte Carlo prices added.")

In [ ]:
# === Chunk N: Model errors vs market ===

demo_df['mc_error']  = demo_df['mc_price']       - demo_df['mid']
demo_df['bs_error']  = demo_df['bs_price']       - demo_df['mid']
demo_df['bin_error'] = demo_df['binomial_price'] - demo_df['mid']

def mae(x):
    return float(np.nanmean(np.abs(x)))

print("MAE (BS)  :", mae(demo_df['bs_error']))
print("MAE (BIN) :", mae(demo_df['bin_error']))
print("MAE (MC)  :", mae(demo_df['mc_error']))

In [ ]:
# === Chunk O: Plot MC error vs strike ===

import plotly.express as px

plot_df = demo_df.dropna(subset=['mc_error'])

fig = px.scatter(
    plot_df,
    x='strike',
    y='mc_error',
    color='mid',
    size='mid',
    title="Monte Carlo Pricing Error vs Market",
    labels={'mc_error': 'Pricing Error', 'strike': 'Strike'}
)

fig.show()
safe_write_html(fig, "mc_error_plot.html")

In [ ]:
# === Chunk P: Convergence check (paths vs abs error) ===

import plotly.graph_objects as go

paths_list = [5_000, 10_000, 25_000, 50_000, 100_000, 200_000]

sample = demo_df.iloc[10]
S = sample['spot']; K = sample['strike']; T = sample['T']; r = sample['r']
sigma = sample['iv_calc']; opt_type = sample['type']
bs_val = bs_price(S, K, T, r, sigma, option_type=opt_type)

errs = []
ses  = []

for n in paths_list:
    price, se = mc_price(S, K, T, r, sigma, n_paths=n, option_type=opt_type, seed=11)
    errs.append(abs(price - bs_val))
    ses.append(se)

fig = go.Figure()
fig.add_trace(go.Scatter(x=paths_list, y=errs, mode='lines+markers', name='|MC - BS|'))
fig.add_trace(go.Scatter(x=paths_list, y=ses,  mode='lines+markers', name='Std Error'))

fig.update_layout(title="Monte Carlo Convergence (paths vs error)",
                  xaxis_title="Number of paths",
                  yaxis_title="Error")

fig.show()
safe_write_html(fig, "mc_convergence.html")